# Random Forest Classifier on Fink/LSST Simulations

- **Author**: Sylvie Dagoret-Campagne
- **Affiliation**: IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Creation date**: 2026-04-13
- **Kernel**: conda_py313

## Goal

Build a simple Random Forest classifier on the ELAsTiCC2 LSST simulations
provided in the `sims/` folder of this tutorial.

The simulated transient classes available are:
- AGN
- KN (Kilonova)
- SLSN
- SNII
- SNIIn
- SNIa
- SNIb
- SNIc
- TDE

## Strategy

1. Read all FITS files using the robust `read_fits` function from `inspect_lcs_asbroker.ipynb`
   (which correctly handles the SNID assignment and separator rows `MJD == -777`).
2. Extract per-object features from the photometry:
   - Per band: `flux_max`, `flux_min`, `flux_mean`, `flux_std`, `mjd_peak`, `rise_time`
   - Global: `NOBS`, `REDSHIFT_FINAL`
3. Train a `RandomForestClassifier` (scikit-learn).
4. Evaluate with classification report and confusion matrix.


## 1 — Imports

In [ ]:
import re
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import astropy.units as u
from astropy.table import Table
from astropy.coordinates import SkyCoord

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

print('Imports OK')

## 2 — Utility functions (adapted from inspect_lcs_asbroker.ipynb)

In [ ]:
def convert_fluxcal_to_mag(df):
    """Convert FLUXCAL to AB magnitude (zero-point 27.5) and add magnitude error."""
    df = df.copy()
    flux = df['FLUXCAL'].copy()
    flux[flux <= 0] = np.nan
    df['magnitude'] = -2.5 * np.log10(flux) + 27.5
    df['magnitude_error'] = 2.5 / np.log(10) * df['FLUXCALERR'] / df['FLUXCAL']
    return df


def tag_DDF(df_header):
    """Tag DDF fields in the header DataFrame (3-degree radius approximation)."""
    dff_coords_dic = {
        'Field':   ['ELAISS1', 'XMM_LSS', 'ECDFS',  'COSMOS', 'EDFS_a', 'EDFS_b'],
        'RA':      [9.45,      35.57,     52.98,    150.11,    58.9,     63.6],
        'DEC':     [-44.02,    -4.82,     -28.12,     2.23,   -49.32,   -47.6],
    }
    ddf_fields = pd.DataFrame(dff_coords_dic)

    df_header['in_ddf_field'] = False
    df_header['ddf_field_name'] = pd.Series(dtype='object')

    coords_header = SkyCoord(
        ra=df_header['RA'].values * u.deg,
        dec=df_header['DEC'].values * u.deg
    )
    for _, row in ddf_fields.iterrows():
        field_coord = SkyCoord(ra=row['RA'] * u.deg, dec=row['DEC'] * u.deg)
        separation = coords_header.separation(field_coord)
        within = separation <= 3 * u.deg
        df_header.loc[within, 'in_ddf_field'] = True
        df_header.loc[within, 'ddf_field_name'] = row['Field']
    return df_header


def read_fits(fname, drop_separators=False):
    """
    Load SNANA formatted PHOT.FITS(.gz) + HEAD.FITS(.gz) and return two DataFrames.

    Parameters
    ----------
    fname : str
        Path to the PHOT.FITS(.gz) file.
    drop_separators : bool
        If True, rows with MJD == -777 are removed from the photometry DataFrame.

    Returns
    -------
    df_header : pd.DataFrame  — header metadata (one row per object)
    df_phot   : pd.DataFrame  — photometry (multiple rows per object)
    """
    # Load photometry
    dat = Table.read(fname, format='fits')
    df_phot = dat.to_pandas()

    # Remove outer sentinel rows
    if df_phot['MJD'].values[-1] == -777.0:
        df_phot = df_phot.drop(df_phot.index[-1])
    if df_phot['MJD'].values[0] == -777.0:
        df_phot = df_phot.drop(df_phot.index[0])

    # Load header
    header = Table.read(fname.replace('PHOT', 'HEAD'), format='fits')
    df_header = header.to_pandas()
    df_header['SNID'] = df_header['SNID'].astype(np.int32)

    # Tag source type from directory name
    folder = os.path.basename(os.path.dirname(fname))
    df_header['SN_TYPE_FROM_PATH'] = folder

    # Assign SNID to each photometric row
    arr_ID = np.zeros(len(df_phot), dtype=np.int32)
    arr_idx = np.where(df_phot['MJD'].values == -777.0)[0]
    arr_idx = np.hstack((np.array([0]), arr_idx, np.array([len(df_phot)])))
    for counter in range(1, len(arr_idx)):
        start, end = arr_idx[counter - 1], arr_idx[counter]
        arr_ID[start:end] = df_header['SNID'].iloc[counter - 1]
    df_phot['SNID'] = arr_ID

    if drop_separators:
        df_phot = df_phot[df_phot['MJD'] != -777.0]

    # Signal-to-noise ratio
    df_phot['SNR'] = df_phot['FLUXCAL'] / df_phot['FLUXCALERR']

    # Latest MJD per SNID with SNR > 5
    df_max_mjd = (
        df_phot[df_phot['SNR'] > 5]
        .groupby('SNID', as_index=False)['MJD']
        .max()
        .rename(columns={'MJD': 'max_MJD_SNR_gt_5'})
    )
    df_phot = df_phot.merge(df_max_mjd, on='SNID')

    # Convert BAND bytes -> string
    if df_phot['BAND'].dtype == object:
        try:
            df_phot['BAND'] = df_phot['BAND'].str.decode('utf-8').str.strip()
        except AttributeError:
            df_phot['BAND'] = df_phot['BAND'].astype(str).str.strip()
    else:
        df_phot['BAND'] = df_phot['BAND'].astype(str).str.strip()

    # Convert to magnitude
    df_phot = convert_fluxcal_to_mag(df_phot)

    # Tag DDF in header
    df_header = tag_DDF(df_header)

    return df_header, df_phot


print('Functions defined OK')

## 3 — Read all simulations

In [ ]:
df_header_list = []
df_phot_list   = []

list_to_read = glob.glob('sims/*/*PHOT.FITS.gz')
print(f'Found {len(list_to_read)} PHOT.FITS.gz files to read.')

for fphot in sorted(list_to_read):
    print(f'  Reading {fphot}')
    df_header_tmp, df_phot_tmp = read_fits(fphot, drop_separators=True)
    df_header_list.append(df_header_tmp)
    df_phot_list.append(df_phot_tmp)

df_header = pd.concat(df_header_list, ignore_index=True)
df_phot   = pd.concat(df_phot_list,   ignore_index=True)

print()
print(f'Total light curves : {len(df_header)}')
print(f'Total photometric points : {len(df_phot)}')

# Summary per class
total_counts = df_header.groupby('SN_TYPE_FROM_PATH')['SNID'].count()
ddf_counts   = df_header[df_header['in_ddf_field']].groupby('SN_TYPE_FROM_PATH')['SNID'].count()
summary_df   = pd.DataFrame({'Total': total_counts, 'In DDF': ddf_counts.fillna(0).astype(int)})
summary_df['% in DDF'] = (summary_df['In DDF'] / summary_df['Total'] * 100).round(1)
print()
print(summary_df)

## 4 — Feature extraction

For each object we compute per-band features:
- `flux_max`, `flux_min`, `flux_mean`, `flux_std` — simple flux statistics
- `mjd_peak` — date of maximum flux
- `rise_time` — time interval above 50 % of flux_max (proxy for width)

And global features:
- `NOBS` — number of observations (from header)
- `REDSHIFT_FINAL` — spectroscopic or photometric redshift

In [ ]:
LSST_BANDS = ['u', 'g', 'r', 'i', 'z', 'Y']


def extract_features_per_object(snid, df_phot_obj, df_header_obj):
    """Extract a feature dictionary for one transient object."""
    feat = {}

    # Per-band features
    for band in LSST_BANDS:
        band_label = f'LSST-{band}'
        mask = df_phot_obj['BAND'] == band_label
        flux = df_phot_obj.loc[mask, 'FLUXCAL'].values
        mjd  = df_phot_obj.loc[mask, 'MJD'].values

        if len(flux) > 0:
            flux_max  = float(np.max(flux))
            flux_min  = float(np.min(flux))
            flux_mean = float(np.mean(flux))
            flux_std  = float(np.std(flux))
            idx_peak  = int(np.argmax(flux))
            mjd_peak  = float(mjd[idx_peak])

            # Rise time: span of points above half-maximum
            half_max = flux_max / 2.0
            above_half = mjd[flux >= half_max]
            rise_time = float(above_half[-1] - above_half[0]) if len(above_half) >= 2 else 0.0

            nobs_band = int(len(flux))
        else:
            flux_max  = 0.0
            flux_min  = 0.0
            flux_mean = 0.0
            flux_std  = 0.0
            mjd_peak  = 0.0
            rise_time = 0.0
            nobs_band = 0

        b = band  # short key
        feat[f'flux_max_{b}']  = flux_max
        feat[f'flux_min_{b}']  = flux_min
        feat[f'flux_mean_{b}'] = flux_mean
        feat[f'flux_std_{b}']  = flux_std
        feat[f'mjd_peak_{b}']  = mjd_peak
        feat[f'rise_time_{b}'] = rise_time
        feat[f'nobs_{b}']      = nobs_band

    # Global features from header
    feat['NOBS']            = int(df_header_obj['NOBS'].values[0])            if 'NOBS'            in df_header_obj.columns else 0
    feat['REDSHIFT_FINAL']  = float(df_header_obj['REDSHIFT_FINAL'].values[0]) if 'REDSHIFT_FINAL'  in df_header_obj.columns else 0.0

    return feat


# Build feature matrix
features_list = []
labels_list   = []

for snid in df_header['SNID'].unique():
    df_phot_obj  = df_phot[df_phot['SNID'] == snid]
    df_header_obj = df_header[df_header['SNID'] == snid]
    feat  = extract_features_per_object(snid, df_phot_obj, df_header_obj)
    label = df_header_obj['SN_TYPE_FROM_PATH'].values[0]
    features_list.append(feat)
    labels_list.append(label)

X = pd.DataFrame(features_list)
y = np.array(labels_list)

print(f'Feature matrix shape: {X.shape}')
print(f'Class distribution:')
unique, counts = np.unique(y, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f'  {cls:10s}: {cnt}')
print()
print('Feature columns:')
print(X.columns.tolist())

## 5 — Train / Test split

In [ ]:
# Fill NaN values (objects with no detections in some bands)
X = X.fillna(0.0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

## 6 — Train the Random Forest

In [ ]:
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)
print('Training done.')

## 7 — Evaluation

In [ ]:
y_pred = clf.predict(X_test)

print('Classification report:')
print(classification_report(y_test, y_pred))

## 8 — Confusion matrix

In [ ]:
classes = np.unique(y)
cm = confusion_matrix(y_test, y_pred, labels=classes)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
ax.set_title('Confusion Matrix — Random Forest on LSST ELAsTiCC2 simulations')
plt.tight_layout()
plt.savefig('confusion_matrix_RF.pdf', bbox_inches='tight')
plt.savefig('confusion_matrix_RF.png', dpi=150, bbox_inches='tight')
plt.show()

## 9 — Feature importances

In [ ]:
importances = clf.feature_importances_
feat_names  = X.columns.tolist()

sorted_idx = np.argsort(importances)[::-1]
top_n = 20

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(top_n), importances[sorted_idx[:top_n]])
ax.set_xticks(range(top_n))
ax.set_xticklabels([feat_names[i] for i in sorted_idx[:top_n]], rotation=45, ha='right')
ax.set_xlabel('Feature')
ax.set_ylabel('Importance')
ax.set_title(f'Top {top_n} feature importances (Random Forest)')
plt.tight_layout()
plt.savefig('feature_importances_RF.pdf', bbox_inches='tight')
plt.savefig('feature_importances_RF.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features:')
for rank, idx in enumerate(sorted_idx[:10]):
    print(f'  {rank+1:2d}. {feat_names[idx]:25s}  importance={importances[idx]:.4f}')

## 10 — Probability distributions per class

Inspect the probability assigned to the correct class for each object in the test set.

In [ ]:
y_proba      = clf.predict_proba(X_test)          # shape (n_test, n_classes)
class_labels = clf.classes_                        # ordered list of class names

# For each test object, retrieve the predicted probability of the TRUE class
true_class_proba = []
for i, true_label in enumerate(y_test):
    idx = list(class_labels).index(true_label)
    true_class_proba.append(y_proba[i, idx])
true_class_proba = np.array(true_class_proba)

# Plot histogram per true class
fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for k, cls in enumerate(sorted(np.unique(y_test))):
    ax = axes[k]
    mask_cls = y_test == cls
    ax.hist(true_class_proba[mask_cls], bins=20, range=(0, 1),
            color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(cls, fontsize=9)
    ax.set_xlabel('P(true class)', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)

# Hide unused subplots
for k in range(len(np.unique(y_test)), len(axes)):
    axes[k].set_visible(False)

fig.suptitle('Distribution of P(true class) per class — RF test set', fontsize=12)
plt.tight_layout()
plt.savefig('proba_true_class_RF.pdf', bbox_inches='tight')
plt.savefig('proba_true_class_RF.png', dpi=150, bbox_inches='tight')
plt.show()

## 11 — Summary

| Step | Description |
|------|-------------|
| Data reading | `read_fits()` from `inspect_lcs_asbroker.ipynb` — correctly assigns SNID via `MJD == -777` separators |
| Features | 7 per band × 6 bands + 2 global = 44 features |
| Classifier | `sklearn.ensemble.RandomForestClassifier` (200 trees) |
| Evaluation | classification_report + confusion matrix + feature importances |

### Possible improvements
- Add more features: rise/decline rates, colour indices (flux ratios between bands), duration above SNR threshold.
- Use time-series features from `tsfresh` or `cesium`.
- Try other classifiers: Gradient Boosting (XGBoost, LightGBM), SVM, neural networks.
- Apply redshift normalisation (shift MJD so that `mjd_peak` is at rest-frame epoch).
- Cross-validate instead of a single train/test split.
- Restrict to DDF events only.
